In [1]:
from concurrent.futures import ThreadPoolExecutor
from contextlib import closing
from pathlib import Path
from time import perf_counter

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from model import FireForecast, MemoryAdapter, RolloutSpatialCache

torch.set_num_threads(4)  # Avoid oversubscribing CPU tensor preparation.


In [2]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_dir=Path("data")
learning_rate=3e-5
low_vram=True  # Checkpoint daily CNN work while retaining a GPU feature cache.
gpu_cache_gb=4.0  # GPU feature cache; inputs are also cached per fire.
use_bf16=device.type=="cuda" and torch.cuda.is_bf16_supported()
history_days=None  # None: all observations since the start of this fire.
rollout_steps=32  # Predict at most 32 future days from every starting day.
loss_decay=0.9  # Relative lead weights: 1, 0.9, 0.81, ...
labels=("active", "burned", "new")

class FireDataset(Dataset):
    def __init__(self, files):
        self.files=files
        self.clear_cache()

    def __len__(self):
        return len(self.files)-1

    def read_day(self, index):
        if self.cache is not None:
            return self.cache[index]
        with np.load(self.files[index]) as data:
            day={name: data[name] for name in ("image", "date", *labels)}
        # Prepare once per cached day, rather than rebuilding masks for every history.
        image=day["image"]
        valid=np.isfinite(image)
        day["image"]=np.concatenate([np.where(valid, image, 0), valid], axis=0)
        return day

    def prepare_cache(self):
        # CPU-only: the next fire can be decompressed while this one trains.
        if self.cache is None:
            self.cache=[self.read_day(i) for i in range(len(self.files))]
            self.cpu_images=np.stack([day["image"] for day in self.cache])
            for index, day in enumerate(self.cache):
                day["image"]=self.cpu_images[index]

    def load_cache(self):
        self.prepare_cache()
        self.images=torch.from_numpy(self.cpu_images).to(device=device, dtype=torch.bfloat16 if use_bf16 else torch.float32)
        self.original_images=None if low_vram else self.images.clone()
        self.dates=torch.tensor([day["date"].item() for day in self.cache], device=device, dtype=torch.float32)
        self.targets={name: torch.as_tensor(np.stack([day[name] for day in self.cache]), device=device, dtype=torch.float32) for name in labels}

    def clear_cache(self):
        self.cache=self.cpu_images=self.images=self.original_images=self.dates=self.targets=None

    def __getitem__(self, index):
        stop=index+1
        start=0 if history_days is None else max(0, stop-history_days)
        if self.images is not None:
            history=self.images[start:stop]
            dates=self.dates[start:stop]
            targets={name: values[stop] for name, values in self.targets.items()}
        else:
            days=[self.read_day(i) for i in range(start, stop)]
            history=torch.from_numpy(np.stack([day["image"] for day in days]))
            dates=torch.tensor([day["date"].item() for day in days], dtype=torch.float32)
            target=self.read_day(stop)
            targets={name: torch.tensor(target[name], dtype=torch.float32) for name in labels}
        return history, dates-dates[-1], targets


def prefetch_fires(loaders, order):
    # Keep only the current fire and one upcoming fire in CPU memory.
    with ThreadPoolExecutor(max_workers=1) as pool:
        pending=pool.submit(loaders[order[0]].dataset.prepare_cache)
        try:
            for position, index in enumerate(order):
                pending.result()
                if position+1<len(order):
                    pending=pool.submit(loaders[order[position+1]].dataset.prepare_cache)
                yield loaders[index].dataset
        finally:
            try:
                pending.result()
            finally:
                for index in order:
                    loaders[index].dataset.clear_cache()


In [3]:
# Add new training fire IDs to train_fires.txt; keep validation IDs fixed.
train_fires=Path("train_fires.txt").read_text().splitlines()
val_fires=Path("val_fires.txt").read_text().splitlines()

loader_options=dict(num_workers=0, pin_memory=False)  # Each fire is cached on GPU.

train_loaders=[DataLoader(FireDataset(sorted((data_dir/fire).glob("*.npz"))), batch_size=None, shuffle=False, **loader_options) for fire in train_fires]
val_loaders=[DataLoader(FireDataset(sorted((data_dir/fire).glob("*.npz"))), batch_size=None, shuffle=False, **loader_options) for fire in val_fires]
train_data=train_loaders[0].dataset

In [4]:
memory=MemoryAdapter(high=1.0, low=-1.0, dims=3, size=4096, device=device,shape={"encoder": 512}, steps=8, input_gain=8.0)
network=FireForecast(input_channels=train_data[0][0].shape[1],memory=memory, low_vram=low_vram, gpu_cache_gb=gpu_cache_gb).to(device)
optimizer=torch.optim.AdamW(network.parameters(), lr=learning_rate, weight_decay=1e-4)

# Compile spatial work and memory; allow normal execution if the compile cache fills.
if device.type=="cuda":
    torch.backends.cudnn.benchmark=True  # Select fast convolution kernels for recurring shapes.
    torch._dynamo.config.cache_size_limit=32  # Cache variants for native image sizes and gradient modes.
    # Fixed-size daily encoding for both cache creation and gradient replay.
    network.encoder._encode_day=torch.compile(network.encoder._encode_day, fullgraph=False, dynamic=True)
    network.encoder._aggregate_skip=torch.compile(network.encoder._aggregate_skip, fullgraph=False, dynamic=True)
    network.decoder.forward=torch.compile(network.decoder.forward, fullgraph=False, dynamic=False)
    network.output.forward=torch.compile(network.output.forward, fullgraph=False, dynamic=False)
    memory.memory.connections=torch.compile(memory.memory.connections, fullgraph=True, dynamic=False)
    memory.memory.step=torch.compile(memory.memory.step, fullgraph=False, dynamic=False)


In [ ]:
epochs=50
patience=7
min_delta=0.0001
noimprove=0
best_val_loss=float("inf")
stopping_loss=float("inf")
losses=[]

checkpoint_dir=Path("checkpoints/recursive")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint=torch.load("latest.pt", map_location="cpu", weights_only=True)
network.load_state_dict(checkpoint.get("model", checkpoint))
start_epoch=checkpoint.get("epoch", 29)  # Last completed epoch reported for the legacy weights-only file.
if "optimizer" in checkpoint:
    optimizer.load_state_dict(checkpoint["optimizer"])
    for group in optimizer.param_groups:
        group["lr"]=learning_rate
    if (checkpoint["rollout_steps"]==rollout_steps and checkpoint["history_days"]==history_days
            and checkpoint["loss_decay"]==loss_decay and checkpoint["snn_input_gain"]==network.memory.input_gain):
        best_val_loss=checkpoint["best_val_loss"]
        stopping_loss=checkpoint["stopping_loss"]
        noimprove=checkpoint["noimprove"]
        losses=checkpoint["losses"]
del checkpoint
print(f"Resuming latest.pt; next epoch: {start_epoch+1}.")

In [6]:
memory=network.memory.memory
memory.voltage=torch.zeros_like(memory.voltage)
memory.spikes=torch.zeros_like(memory.spikes)
memory.y={key: memory.voltage.new_zeros(width) for key, width in memory.shape.items()}

for epoch in range(start_epoch, epochs):
    if noimprove>=patience:
        break
    epoch_started=perf_counter()
    print(f"Epoch {epoch+1}/{epochs}:")
    for phase, loaders in (("train", train_loaders), ("val", val_loaders)):
        training=phase=="train"
        network.train(training)
        total_loss=torch.zeros((), device=device)
        samples=0
        order=np.random.permutation(len(loaders)) if training else range(len(loaders))

        with closing(prefetch_fires(loaders, order)) as fires, torch.set_grad_enabled(training):
            for dataset in fires:
                try:
                    dataset.load_cache()
                    for origin in range(len(dataset)):
                        steps=min(rollout_steps, len(dataset)-origin)
                        weight_sum=sum(loss_decay**lead for lead in range(steps))
                        if training:
                            optimizer.zero_grad(set_to_none=True)
                        changed_days=[]
                        observed_memory=None
                        spatial_cache=RolloutSpatialCache(network.encoder, capacity=origin+steps) if history_days is None else None
                        try:
                            for step in range(steps):
                                history, dates, targets=dataset[origin+step]
                                history=history.unsqueeze(0)
                                dates=dates.unsqueeze(0).to(device)
                                targets={name: target.unsqueeze(0).to(device) for name, target in targets.items()}

                                memory.voltage=memory.voltage.detach()
                                memory.spikes=memory.spikes.detach()
                                memory.y={key: value.detach() for key, value in memory.y.items()}

                                with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=use_bf16):
                                    active, burned, newly_burned=network(history, dates, spatial_cache=spatial_cache)
                                loss=F.binary_cross_entropy(active, targets["active"])+ F.binary_cross_entropy(burned, targets["burned"])+ 0.5*F.binary_cross_entropy(newly_burned, targets["new"])
                                weight=loss_decay**step/weight_sum
                                loss=loss*weight
                                if training:
                                    loss.backward()  # Free this step's graph; accumulate weighted gradients.
                                total_loss+=loss.detach()
                                samples+=weight

                                if step==0:
                                    # Save observed-prefix memory before simulating future days.
                                    observed_memory=(memory.voltage.detach().clone(), memory.spikes.detach().clone(),
                                                     {key: value.detach().clone() for key, value in memory.y.items()})
                                if step+1<steps:
                                    target_day=origin+step+1
                                    # Reuse the cache to avoid copying the entire growing history.
                                    changed_days.append(target_day)
                                    with torch.no_grad():
                                        dataset.images[target_day].copy_(dataset.images[origin])
                                        dataset.images[target_day, 27:28].copy_(active.detach()[0])
                                        dataset.images[target_day, 28:29].copy_(burned.detach()[0])
                                        dataset.images[target_day, 56:58]=1  # Generated fire channels are valid.
                                del history, dates, targets, active, burned, newly_burned, loss
                            if training:
                                if spatial_cache is not None:
                                    spatial_cache.backward()
                                torch.nn.utils.clip_grad_norm_(network.parameters(), 1.0)
                                optimizer.step()
                        finally:
                            if spatial_cache is not None:
                                spatial_cache.clear()
                            # Next origin uses real observations and its observed-prefix memory.
                            for day in changed_days:
                                # Restore on GPU when space permits; low-VRAM mode uses the CPU copy.
                                original=dataset.original_images[day] if dataset.original_images is not None else torch.from_numpy(dataset.cache[day]["image"])
                                dataset.images[day].copy_(original)
                            changed_days.clear()
                            if observed_memory is not None:
                                memory.voltage, memory.spikes, memory.y=observed_memory
                finally:
                    dataset.clear_cache()

                memory.voltage=torch.zeros_like(memory.voltage)
                memory.spikes=torch.zeros_like(memory.spikes)
                memory.y={key: memory.voltage.new_zeros(width) for key, width in memory.shape.items()}

        average_loss=(total_loss/samples).item()
        print(f"{phase} loss: {average_loss:.6f}")
        if phase=="val":
            val_loss=average_loss

    losses.append(val_loss)
    if val_loss<best_val_loss:
        best_val_loss=val_loss
        torch.save(network.state_dict(), "best_recursive.pt")
    if val_loss<stopping_loss-min_delta:
        stopping_loss=val_loss
        noimprove=0
    else:
        noimprove+=1
        print(f"Early stopping: {noimprove}/{patience} epochs without improvement.")

    checkpoint={
        "epoch": epoch+1,
        "history_days": history_days,
        "rollout_steps": rollout_steps,
        "loss_decay": loss_decay,
        "snn_input_gain": network.memory.input_gain,
        "feedback": "probability",
        "model": network.state_dict(),
        "optimizer": optimizer.state_dict(),
        "best_val_loss": best_val_loss,
        "stopping_loss": stopping_loss,
        "noimprove": noimprove,
        "losses": losses,
    }
    paths=[Path("latest.pt")]
    if (epoch+1)%5==0:
        paths.append(checkpoint_dir / f"epoch_{epoch+1:03d}.pt")
    for path in paths:
        torch.save(checkpoint, path.with_suffix(".tmp"))
        path.with_suffix(".tmp").replace(path)
    del checkpoint
    print(f"Saved latest.pt after epoch {epoch+1}.")

    print(f"Epoch {epoch+1} took {(perf_counter()-epoch_started)/60:.1f} minutes.")
    if noimprove>=patience:
        print("Early stopped.")
        break


Epoch 21/50:


W1115 00:02:23.709000 399812 site-packages/torch/_inductor/utils.py:1806] [0/1] Not enough SMs to use max_autotune_gemm mode
[W1115 00:06:56.676738795 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 2099249152 bytes (free: 196083712, total: 12346195968).
[W1115 00:07:09.164880753 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 639631360 bytes (free: 336592896, total: 12346195968).
[W1115 00:07:15.383864042 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 803209216 bytes (free: 80740352, total: 12346195968).
[W1115 00:07:20.817774410 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 926941184 bytes (free: 177930240, total: 12346195968).
[W1115 00:07:24.763216641 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 1008730112 byte

train loss: 0.0152
val loss: 0.0110
Epoch 21 took 14.9 minutes.
Epoch 22/50:
train loss: 0.0145
val loss: 0.0111
Early stopping: 1/7 epochs without improvement.
Epoch 22 took 12.8 minutes.
Epoch 23/50:
train loss: 0.0145
val loss: 0.0110
Early stopping: 2/7 epochs without improvement.
Epoch 23 took 12.5 minutes.
Epoch 24/50:
train loss: 0.0140
val loss: 0.0113
Early stopping: 3/7 epochs without improvement.
Epoch 24 took 12.5 minutes.
Epoch 25/50:
train loss: 0.0140
val loss: 0.0108
Early stopping: 4/7 epochs without improvement.
Saved checkpoints/epoch_025.pt
Epoch 25 took 12.7 minutes.
Epoch 26/50:
train loss: 0.0144
val loss: 0.0109
Early stopping: 5/7 epochs without improvement.
Epoch 26 took 12.6 minutes.
Epoch 27/50:
train loss: 0.0137
val loss: 0.0108
Early stopping: 6/7 epochs without improvement.
Epoch 27 took 12.6 minutes.
Epoch 28/50:
train loss: 0.0136
val loss: 0.0105
Early stopping: 7/7 epochs without improvement.
Epoch 28 took 12.4 minutes.
Early stopped.
